In [1]:
from transformers import AutoModelForCausalLM

In [2]:
model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b")

You have loaded an FP4 model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model. To remove this warning, pass device_map = 'cuda'. 


Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

__init__.py:   0%|          | 0.00/179 [00:00<?, ?B/s]

matmul_ogs.py: 0.00B [00:00, ?B/s]

_finalize_matmul.py: 0.00B [00:00, ?B/s]

...

hopper_value.py: 0.00B [00:00, ?B/s]

_topk_backward.py: 0.00B [00:00, ?B/s]

_topk_forward.py: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

In [3]:
# Print layer structure


layer = model.model.layers[17]

print("Layer structure:")
for name, module in layer.named_children():
    print(f"  {name}: {type(module).__name__}")
    if hasattr(module, 'weight'):
        print(f"    - weight shape: {module.weight.shape}")
    if hasattr(module, 'bias') and module.bias is not None:
        print(f"    - bias shape: {module.bias.shape}")
print()

Layer structure:
  self_attn: GptOssAttention
  mlp: GptOssMLP
  input_layernorm: GptOssRMSNorm
    - weight shape: torch.Size([2880])
  post_attention_layernorm: GptOssRMSNorm
    - weight shape: torch.Size([2880])



In [4]:
layer.mlp.router

GptOssTopKRouter()

In [5]:
import torch
import torch.nn as nn
from typing import Any, Set, Union

def print_tree_structure(obj, name="root", prefix="", visited=None, max_depth=10, current_depth=0):
    """
    Print object attributes in a tree-like structure
    
    Args:
        obj: Object to inspect
        name: Name of the current object
        prefix: Current indentation prefix
        visited: Set of visited object IDs to avoid infinite recursion
        max_depth: Maximum depth to traverse
        current_depth: Current traversal depth
    """
    if visited is None:
        visited = set()
    
    # Avoid infinite recursion
    obj_id = id(obj)
    if obj_id in visited or current_depth >= max_depth:
        if current_depth >= max_depth:
            print(f"{prefix}├── {name}: <max depth reached>")
        else:
            print(f"{prefix}├── {name}: <already visited>")
        return
    
    visited.add(obj_id)
    
    # Print current object info
    obj_type = type(obj).__name__
    obj_info = get_object_info(obj)
    print(f"{prefix}├── {name}: {obj_type}{obj_info}")
    
    # Get all attributes (including properties, methods, etc.)
    try:
        attrs = []
        
        # Get regular attributes
        if hasattr(obj, '__dict__'):
            attrs.extend([(k, v) for k, v in obj.__dict__.items() 
                         if not k.startswith('_')])
        
        # For PyTorch modules, also get named children and parameters
        if isinstance(obj, nn.Module):
            # Named children (submodules)
            for child_name, child_module in obj.named_children():
                if child_name not in [k for k, v in attrs]:
                    attrs.append((child_name, child_module))
            
            # Named parameters that aren't in children
            for param_name, param in obj.named_parameters(recurse=False):
                if param_name not in [k for k, v in attrs]:
                    attrs.append((param_name, param))
            
            # Named buffers
            for buffer_name, buffer in obj.named_buffers(recurse=False):
                if buffer_name not in [k for k, v in attrs]:
                    attrs.append((buffer_name, buffer))
        
        # Sort attributes alphabetically
        attrs.sort(key=lambda x: x[0])
        
        # Print each attribute
        for i, (attr_name, attr_value) in enumerate(attrs):
            is_last = (i == len(attrs) - 1)
            new_prefix = prefix + ("    " if is_last else "│   ")
            
            print_tree_structure(
                attr_value, 
                attr_name, 
                new_prefix, 
                visited.copy(),  # Pass a copy to allow revisiting in different branches
                max_depth, 
                current_depth + 1
            )
    
    except Exception as e:
        print(f"{prefix}    └── <error accessing attributes: {e}>")

def get_object_info(obj) -> str:
    """Get relevant info about an object for display"""
    info_parts = []
    
    # For tensors
    if isinstance(obj, torch.Tensor):
        info_parts.append(f"shape={tuple(obj.shape)}")
        info_parts.append(f"dtype={obj.dtype}")
        if obj.device.type != 'cpu':
            info_parts.append(f"device={obj.device}")
        if obj.requires_grad:
            info_parts.append("requires_grad=True")
    
    # For nn.Module
    elif isinstance(obj, nn.Module):
        # Count parameters
        try:
            param_count = sum(p.numel() for p in obj.parameters())
            if param_count > 0:
                info_parts.append(f"params={param_count:,}")
        except:
            pass
        
        # Check if it has weight
        if hasattr(obj, 'weight') and obj.weight is not None:
            info_parts.append(f"weight_shape={tuple(obj.weight.shape)}")
        
        # Check common attributes
        if hasattr(obj, 'in_features') and hasattr(obj, 'out_features'):
            info_parts.append(f"in_features={obj.in_features}")
            info_parts.append(f"out_features={obj.out_features}")
        elif hasattr(obj, 'num_features'):
            info_parts.append(f"num_features={obj.num_features}")
    
    # For basic types
    elif isinstance(obj, (int, float, str, bool)):
        if isinstance(obj, str) and len(obj) > 50:
            info_parts.append(f"'{obj[:47]}...'")
        else:
            info_parts.append(f"= {repr(obj)}")
    
    # For collections
    elif isinstance(obj, (list, tuple)):
        info_parts.append(f"len={len(obj)}")
    elif isinstance(obj, dict):
        info_parts.append(f"keys={len(obj)}")
    
    return f" ({', '.join(info_parts)})" if info_parts else ""

def inspect_mlp_tree(layer, mlp_attr_name='mlp', max_depth=8):
    """
    Inspect MLP component in tree structure
    
    Args:
        layer: The transformer layer object
        mlp_attr_name: Name of the MLP attribute (e.g., 'mlp', 'ffn', 'feed_forward')
        max_depth: Maximum depth to traverse
    """
    print("="*80)
    print(f"MLP TREE STRUCTURE INSPECTION")
    print("="*80)
    
    # Check if the MLP attribute exists
    if not hasattr(layer, mlp_attr_name):
        print(f"Error: Layer does not have attribute '{mlp_attr_name}'")
        print("Available attributes:")
        for attr in sorted(dir(layer)):
            if not attr.startswith('_'):
                print(f"  - {attr}")
        return None
    
    mlp = getattr(layer, mlp_attr_name)
    
    print(f"Inspecting: layer.{mlp_attr_name}")
    print(f"Type: {type(mlp).__name__}")
    print(f"Max depth: {max_depth}")
    print()
    
    # Print the tree structure
    print_tree_structure(mlp, mlp_attr_name, "", max_depth=max_depth)
    
    return mlp

def find_mlp_components(layer):
    """
    Find all possible MLP-like components in a layer
    """
    print("SEARCHING FOR MLP COMPONENTS")
    print("-" * 40)
    
    mlp_names = ['mlp', 'ffn', 'feed_forward', 'fc', 'intermediate']
    found_components = []
    
    for attr_name in dir(layer):
        if not attr_name.startswith('_'):
            attr_value = getattr(layer, attr_name)
            if isinstance(attr_value, nn.Module):
                # Check if it's likely an MLP component
                if any(mlp_name in attr_name.lower() for mlp_name in mlp_names):
                    found_components.append(attr_name)
                    print(f"✓ Found MLP-like component: {attr_name} ({type(attr_value).__name__})")
    
    if not found_components:
        print("No obvious MLP components found. Showing all nn.Module attributes:")
        for attr_name in sorted(dir(layer)):
            if not attr_name.startswith('_'):
                attr_value = getattr(layer, attr_name)
                if isinstance(attr_value, nn.Module):
                    print(f"  - {attr_name}: {type(attr_value).__name__}")
    
    return found_components

# Example usage functions
def quick_mlp_inspect(model, layer_idx=0, mlp_name='mlp'):
    """Quick inspection of MLP in a specific layer"""
    # Try to find layers in different model architectures
    if hasattr(model, 'transformer'):
        layers = model.transformer.h
    elif hasattr(model, 'model') and hasattr(model.model, 'layers'):
        layers = model.model.layers
    else:
        print("Could not find layers automatically. Please specify the path.")
        return None
    
    if layer_idx >= len(layers):
        print(f"Layer index {layer_idx} out of range. Max: {len(layers)-1}")
        return None
    
    layer = layers[layer_idx]
    
    # First, find available MLP components
    components = find_mlp_components(layer)
    print()
    
    # Inspect the specified or first found component
    if mlp_name in [getattr(layer, attr, None) for attr in dir(layer)]:
        return inspect_mlp_tree(layer, mlp_name)
    elif components:
        print(f"'{mlp_name}' not found. Inspecting first found component: '{components[0]}'")
        return inspect_mlp_tree(layer, components[0])
    else:
        print("No MLP components found to inspect.")
        return None

In [6]:
mlp = inspect_mlp_tree(layer, 'mlp', max_depth=8)

MLP TREE STRUCTURE INSPECTION
Inspecting: layer.mlp
Type: GptOssMLP
Max depth: 8

├── mlp: GptOssMLP (params=368,672)
│   ├── experts: Mxfp4GptOssExperts (params=276,480)
│   │   ├── alpha: float (= 1.702)
│   │   ├── down_proj: Tensor
│   │   │   ├── dtype: FloatType
│   │   │   │   ├── bitwidth: int (= 4)
│   │   │   │   ├── bitwidth_exponent: int (= 2)
│   │   │   │   ├── bitwidth_mantissa: int (= 1)
│   │   │       ├── is_signed: bool (= True)
│   │   │   ├── shape: Size (len=3)
│   │   │   ├── shape_max: list (len=3)
│   │       ├── storage: Storage
│   │       │   ├── data: Tensor (shape=(32, 5760, 720), dtype=torch.uint8, device=cuda:0)
│   │           ├── layout: HopperMXValueLayout
│   │           │   ├── K: int (= 1440)
│   │           │   ├── N: int (= 2880)
│   │           │   ├── initial_shape: Size (len=3)
│   │           │   ├── leading_shape: list (len=1)
│   │           │   ├── mma_version: int (= 3)
│   │               ├── mx_axis: int (= 1)
│   │   ├── down_proj_bias

In [11]:
print(mlp.experts.down_proj)

Tensor(storage=Storage(data=tensor([[[108, 232, 178,  ..., 114, 138,  95],
         [248,  85, 151,  ..., 162,  84, 133],
         [ 28, 146, 146,  ...,  72,  97,  12],
         ...,
         [180,  69,  19,  ...,  50,  64, 249],
         [  4, 219, 226,  ...,  34, 236, 238],
         [ 97, 176, 171,  ...,  71, 200, 135]],

        [[ 97, 141,  76,  ..., 233,   3, 109],
         [230,  50,  19,  ..., 238, 248, 240],
         [201, 162,  29,  ..., 113, 168, 137],
         ...,
         [216, 176,  66,  ...,  74, 150,   8],
         [ 85, 139,  85,  ..., 157, 152, 170],
         [ 97, 150, 149,  ..., 241,  58,   7]],

        [[ 23,  84, 172,  ...,  12, 135, 142],
         [134, 134, 152,  ..., 112,  40,  52],
         [ 43,  25,  45,  ...,  17, 130, 217],
         ...,
         [180, 154, 226,  ..., 194, 216, 178],
         [139, 104, 206,  ..., 228,  10,  26],
         [187, 138, 183,  ..., 120,  96, 197]],

        ...,

        [[ 86,  74, 128,  ...,  82,  85,  85],
         [179,  9